# Lancer l'application d'aide au tri (prototype de recherche)

À ouvrir avec le noyau **`memoire_cnn`**, puis exécuter les cellules **dans l'ordre**. Prérequis : notebooks 15 et 16 terminés (c'est le cas).

- L'application tourne **sur cet ordinateur seulement** (adresse `http://127.0.0.1:7860`) : aucune radio ne part sur internet.
- Elle tourne dans un processus séparé : le notebook reste utilisable pendant ce temps.
- Pour l'arrêter : dernière cellule.

In [ ]:
# 1. Réglages et contrôles
import os, socket, subprocess, sys, time
from pathlib import Path

APP = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention\application_clinique")
SYSTEME = "v4:CNN_CBAM"   # autres choix : "v4:CNN_classique", "v3:CNN_CBAM", "v3:CNN_classique"
PORT = 7860
ENV = {**os.environ, "PYTHONIOENCODING": "utf-8"}

for nom in ["application_clinique.py", "outils_clinique.py", "tri_parametres.json"]:
    if not (APP / nom).is_file():
        raise RuntimeError(f"{APP / nom} est introuvable.")
print("Python :", sys.version.split()[0], "|", sys.executable)
print("Dossier de l'application :", APP)

## 2. Installer Gradio (une seule fois)

La cellule ne fait rien si Gradio est déjà installé. Sinon, elle installe **Gradio 6.29.0** (la version testée) en imposant à pip les versions déjà installées de PyTorch, NumPy, pandas, etc. : les notebooks 01 à 16 restent reproductibles. Compter quelques minutes. Si pip refuse (conflit de versions), ne rien forcer : m'envoyer le message affiché.

In [ ]:
# 2. Installation de Gradio (ne fait rien s'il est déjà là)
from importlib import metadata

def version_gradio():
    r = subprocess.run([sys.executable, "-c", "import gradio; print(gradio.__version__)"],
                       capture_output=True, text=True, encoding="utf-8", errors="replace", env=ENV)
    return r.stdout.strip() if r.returncode == 0 else None

deja = version_gradio()
if deja:
    print("Gradio déjà installé :", deja)
else:
    garder = []
    for nom in ["torch", "torchvision", "torchxrayvision", "numpy", "pandas", "pillow", "scipy", "scikit-learn", "matplotlib"]:
        try:
            garder.append(f"{nom}=={metadata.version(nom)}")
        except metadata.PackageNotFoundError:
            pass
    print("Installation de gradio 6.29.0 ; versions imposées :", ", ".join(garder))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "gradio==6.29.0", *garder],
                       capture_output=True, text=True, encoding="utf-8", errors="replace", env=ENV)
    print(r.stdout[-2000:], r.stderr[-2000:])
    if r.returncode != 0:
        raise RuntimeError("L'installation a échoué : copier le message ci-dessus.")
    print("Gradio installé :", version_gradio())

## 3. Lancer l'application

Le chargement des trois modèles (et du segmenteur des poumons pour v4) prend de quelques secondes à une minute. Le navigateur s'ouvre ensuite tout seul ; sinon, ouvrir à la main l'adresse affichée. Les messages de l'application sont dans `application.log`.

In [ ]:
# 3. Lancement
def port_occupe(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False

if port_occupe(PORT):
    raise RuntimeError(f"Une application tourne déjà sur http://127.0.0.1:{PORT} : l'ouvrir dans le navigateur, "
                       "ou l'arrêter (cellule 5), ou choisir un autre PORT dans la cellule 1.")
JOURNAL = APP / "application.log"
journal = open(JOURNAL, "w", encoding="utf-8")
application = subprocess.Popen([sys.executable, str(APP / "application_clinique.py"), "--systeme", SYSTEME, "--port", str(PORT)],
                               cwd=APP, stdout=journal, stderr=subprocess.STDOUT, env=ENV)
debut = time.time()
while True:
    if application.poll() is not None:
        journal.close()
        print("L'application s'est arrêtée. Fin de application.log :\n")
        print(JOURNAL.read_text(encoding="utf-8", errors="replace")[-3000:])
        break
    if port_occupe(PORT):
        print(f"Application prête en {time.time() - debut:.0f} s : http://127.0.0.1:{PORT}  (système {SYSTEME})")
        break
    if time.time() - debut > 300:
        print(f"Pas prête après 5 minutes : lire {JOURNAL}")
        break
    time.sleep(2)

## 4. Utiliser l'application

1. Déposer une radiographie de face (JPEG ou PNG) dans le cadre de gauche. Facultatif : une **référence** (numéro d'examen, code), jamais le nom du patient. Cliquer sur **Analyser**.
2. Lire la fiche : zone (verte, orange ou rouge), score, seuils, alertes. Le bandeau rouge « seuils du développement, non calibrés localement » est **normal** tant qu'aucune calibration locale n'a été faite.
3. En dessous : la carte Grad-CAM (contour jaune : poumons) et l'attention spatiale du CBAM, chacune avec son échelle de couleur, puis l'aide à leur lecture.
4. **Enregistrer l'analyse** garde un rapport PDF, la radio, les cartes et les données dans `application_clinique\analyses`. L'onglet **Historique** les liste : filtre par zone, recherche par référence, réouverture d'une analyse (clic sur une ligne), export CSV pour Excel, suppression.

Pour les captures du mémoire, prendre des radios du dossier `archive\chest_xray\test\NORMAL` et `...\test\PNEUMONIA` : elles n'ont servi ni à l'entraînement, ni au réglage des seuils. Le rapport PDF de chaque analyse enregistrée peut aussi servir de figure.

In [ ]:
# 5. Arrêter l'application
try:
    application.terminate()
    application.wait(timeout=30)
    journal.close()
    print("Application arrêtée.")
except NameError:
    print("Aucune application lancée depuis ce noyau. Si le noyau a été redémarré, l'ancienne tourne peut-être encore : "
          "l'utiliser telle quelle, ou changer PORT dans la cellule 1.")